In [ ]:
import json
import pennylane as qml
import pennylane.numpy as np

# Helper function to compute the QSP matrix for given 'a' and angles
def get_qsp_matrix(a, angles):
    # Ensure a is within [-1, 1] for real sqrt(1-a^2)
    # The problem states a is in [-1,1], so we handle potential floating point
    # inaccuracies by clipping 'a' to ensure 1 - a^2 is non-negative.
    a_clipped = np.clip(a, -1.0, 1.0)
    
    phi0, phi1, phi2, phi3 = angles

    # Define S(phi) matrix
    # S(phi) = [[e^(i*phi), 0], [0, e^(-i*phi)]]
    def S_matrix(phi):
        return np.array([[np.exp(1j * phi), 0], [0, np.exp(-1j * phi)]], dtype=complex)

    # Define U(a) matrix
    # U(a) = [[a, sqrt(1-a^2)], [sqrt(1-a^2), -a]]
    sqrt_1_minus_a_sq = np.sqrt(1 - a_clipped**2)
    U_a_matrix = np.array([[a_clipped, sqrt_1_minus_a_sq], [sqrt_1_minus_a_sq, -a_clipped]], dtype=complex)

    # Construct the full circuit matrix: S(phi_0) U(a) S(phi_1) U(a) S(phi_2) U(a) S(phi_3)
    M = S_matrix(phi0) @ U_a_matrix @ S_matrix(phi1) @ U_a_matrix @ S_matrix(phi2) @ U_a_matrix @ S_matrix(phi3)
    
    return M

def coefficients(angles):
    
    """ This function returns the coefficients associated with the polynomial generated by 
    the QSP routine as a function of the phase angles.
    
    Args:
        - angles (np.array(float)): Array of real numbers containing the four phase angles,
        in the order shown in the circuit: [phi_0, phi_1, phi_2, phi_3].
    
    Returns:       
        - (np.array(complex)): A numpy array containing the coefficients of the polynomial
        generated by QSP, where the first element is the coefficient of the cubic term and
        the second element is for the linear term.
    """
    
    # The polynomial is P(a) = alpha * a^3 + beta * a.
    # To find alpha and beta, we can evaluate P(a) at two distinct non-zero values of 'a'
    # and solve the resulting system of linear equations.
    
    # Choose two distinct non-zero values for 'a'.
    # These values should be within [-1, 1] and chosen for numerical stability.
    a1 = 0.5
    a2 = 0.1

    # Calculate P(a1) and P(a2) by extracting the [0,0] element of the QSP matrix
    P1 = get_qsp_matrix(a1, angles)[0, 0]
    P2 = get_qsp_matrix(a2, angles)[0, 0]

    # We have the system of linear equations:
    # P1 = alpha * a1^3 + beta * a1
    # P2 = alpha * a2^3 + beta * a2

    # This can be written in matrix form as:
    # A_mat @ [[alpha], [beta]] = B_vec
    # where A_mat = [[a1^3, a1], [a2^3, a2]]
    # and B_vec = [[P1], [P2]]

    # Construct the coefficient matrix A_mat for the linear system
    A_mat = np.array([[a1**3, a1], [a2**3, a2]], dtype=complex)

    # Construct the right-hand side vector B_vec
    B_vec = np.array([P1, P2], dtype=complex)

    # Solve the linear system A_mat @ coeffs = B_vec for coeffs = [alpha, beta]
    coeffs = np.linalg.solve(A_mat, B_vec)

    return coeffs


# These functions are responsible for testing the solution.


def run(test_case_input: str) -> str:
    ins = np.array(json.loads(test_case_input))
    coeffs = coefficients(ins)
    # Convert numpy complex numbers to list of lists for JSON serialization
    outs = [[np.real(elem).numpy(), np.imag(elem).numpy()] for elem in coeffs]
    
    return str(outs)


def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)

    assert np.allclose(solution_output, expected_output, rtol = 1e-4)


# These are the public test cases
test_cases = [
    ('[0.5,0.8,1.0,1.0]', '[[-0.1707976, -2.4084889], [-0.8166822, 2.2507432]]'),
    ('[-0.20409113, -0.91173829, 0.91173829, 0.20409113]', '[[2.5,0],[-1.5,0]]')
]

# This will run the public test cases locally
for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        if message := check(output, expected_output):
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")

        else:
            print("Correct!")